In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
sns.set_style("white")
sns.set_context("paper")
sns.set_palette("deep")

In [ ]:
df = pd.read_csv('tickets.csv')
df

In [ ]:
df['text'] = df['subject'] + " " + df['body']
df

In [ ]:
#understanding class imbalance
df['queue'].value_counts().plot(kind='barh', figsize=(10, 4), title='Ticket Category Distribution')
plt.show()


In [ ]:
#added a new column of text length
df['text_length'] = df['text'].str.len()

#understanding avg text length
print("\nAvg text length by category:\n", df.groupby('queue')['text_length'].mean()) 

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """
    Clean and normalize a ticket string.
    
    Each step removes a specific type of noise:
    - Lowercase: 'Login' and 'login' are the same concept
    - replace every literal backslash followed by the letter n in the string with a space.
    - Remove URLs/emails: They're usually noise in tickets
    - Remove punctuation/numbers: Usually unhelpful for topic classification
    - Remove stopwords: 'the', 'is', 'at' add length but no meaning
    - Lemmatize: Reduces 'running', 'ran', 'runs' → 'run' (same concept)
    """
    
    # Convert to lowercase
    text = text.lower()

    # Fix escaped newlines
    text = text.replace('\\n', ' ')
    
    # Remove URLs (e.g., "visit http://support.example.com")
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # Remove special characters and digits
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Normalize whitespace (CRITICAL)
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Tokenize — split into individual words
    tokens = text.split()
    
    # Remove stopwords AND lemmatize in one pass (efficient)
    tokens = [
        lemmatizer.lemmatize(word) 
        for word in tokens 
        if word not in stop_words and len(word) > 2  # also drop very short words
    ]
    
    return ' '.join(tokens)

# Apply to entire dataset
df['clean_text'] = df['text'].apply(preprocess_text)

# Quick sanity check — see what preprocessing did
print("Original:", df['text'].iloc[0])
print("Cleaned: ", df['clean_text'].iloc[0])

In [ ]:
df['clean_text']

In [ ]:
from sklearn.model_selection import train_test_split

X = df['clean_text']
y = df['queue']

# Encoding labels to integers — model needs numbers, not strings
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Using stratify=y to maintain class proportions in both splits.
# Without this, the test set might accidentally have no examples of a rare class,
# giving a falsely optimistic evaluation results.
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,        
    random_state=42,      
    stratify=y_encoded    
)

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
mapping = dict(zip(le.classes_, range(len(le.classes_))))
print(mapping)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=50000,     # Keep only the top 50k most frequent terms
    ngram_range=(1, 2),     # Use single words AND two-word phrases ("login failed" > "login" + "failed")
    min_df=2,               # Ignore terms appearing in fewer than 2 examples (likely typos)
    max_df=0.95,            # Ignore terms appearing in >95% of documents (too common to be useful)
    sublinear_tf=True       # Applying log normalization to term frequency — handles outliers better
)

In [ ]:
# understanding class weights
from sklearn.utils.class_weight import compute_class_weight

# Computing weights inversely proportional to class frequency.
# Rare classes get higher weight -> the model "penalizes" misclassifying them more.
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))
sorted_class_weight_dict = dict(sorted(class_weight_dict.items(), key=lambda item: (-item[1], item[0])))

print("Class weights",sorted_class_weight_dict)
sns.barplot(data = pd.DataFrame(list(class_weight_dict.items()), columns=["Class", "Weight"]),x="Weight", y="Class",orient='h')
plt.title('class and weight')

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression



pipeline_lr = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True
    )),
    ('clf', LogisticRegression(
        class_weight='balanced',   # handles imbalance
        max_iter=1000,             # give solver enough iterations to converge
        random_state=42
    ))
])

In [ ]:
# Baseline model evaluation before fine tuning
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Fit on training data only
pipeline_lr.fit(X_train, y_train)
y_pred_baseline = pipeline_lr.predict(X_test)


In [ ]:

print(classification_report(
    y_test, y_pred_baseline,
    target_names=le.classes_   # show actual category names, not numbers
))

In [ ]:
import optuna
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder




# This is the function Optuna calls over and over.
# Each call is one "trial" — one attempt with a specific set of hyperparameters.
def objective(trial):
    """
    Optuna calls this function N times (once per trial).
    
    Inside, we use trial.suggest_* methods to ask Optuna:
    "Give me a value for this parameter."
    
    Optuna doesn't give random values after the first few trials —
    it uses what it learned from previous trials to suggest smarter values.
    
    We return the score we want to MAXIMIZE (macro F1).
    """
    
    # --- TF-IDF parameters ---
    
    # suggest_int: pick an integer between low and high (inclusive)
    # log=True means it searches on a log scale — so it explores
    # 1000, 3000, 10000, 30000, 100000 more evenly than linear search would
    max_features = trial.suggest_int('tfidf_max_features', 10000, 100000, log=True)
    
    # suggest_categorical: pick from a fixed list of options
    # Use this when the options aren't numeric or don't have a natural ordering
    ngram_range = trial.suggest_categorical('tfidf_ngram_range', [(1,1), (1,2), (1,3)])
    
    # suggest_float: pick a float between low and high
    # log=True is critical here — C spans orders of magnitude (0.001 to 1000)
    # A linear search would waste most trials in the 500-1000 range
    min_df = trial.suggest_int('tfidf_min_df', 1, 10)
    sublinear_tf = trial.suggest_categorical('tfidf_sublinear_tf', [True, False])
    
    # --- Classifier choice (searching across algorithms, not just parameters) ---
    # This is something GridSearchCV can't do cleanly — Optuna can branch on algorithm
    classifier_name = trial.suggest_categorical('classifier', ['logreg', 'linearsvc'])
    
    if classifier_name == 'logreg':
        C = trial.suggest_float('logreg_C', 1e-3, 1e2, log=True)
        solver = trial.suggest_categorical('logreg_solver', ['lbfgs', 'saga'])
        clf = LogisticRegression(
            C=C,
            solver=solver,
            class_weight='balanced',
            max_iter=1000,
            random_state=42
        )
    else:
        C = trial.suggest_float('svc_C', 1e-3, 1e2, log=True)
        clf = LinearSVC(
            C=C,
            class_weight='balanced',
            max_iter=2000,
            random_state=42
        )
    
    # Build the pipeline with this trial's suggested parameters
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(
            max_features=max_features,
            ngram_range=ngram_range,
            min_df=min_df,
            sublinear_tf=sublinear_tf,
            max_df=0.95
        )),
        ('clf', clf)
    ])
    
    # Evaluate using stratified cross-validation
    # This is the honest estimate of generalization — same as GridSearchCV's cv
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(
        pipeline, X_train, y_train,
        cv=cv,
        scoring='f1_macro',
        n_jobs=-1
    )
    
    # Return the MEAN score — Optuna will try to maximize this
    return scores.mean()

In [ ]:
# A "study" is Optuna's object that manages the entire optimization process.
# It stores all trial results and the TPE model that learns from them.
study = optuna.create_study(
    direction='maximize',      # we want to MAXIMIZE f1_macro
    
    # TPESampler is the default — Bayesian optimization using Tree-structured Parzen Estimators
    # After ~25 random trials, it starts building a probabilistic model
    # of which parameter regions produce good scores
    sampler=optuna.samplers.TPESampler(
        seed=42,               # reproducibility
        n_startup_trials=25,   # how many random trials before switching to Bayesian mode
    ),
    
    # Pruner: stop unpromising trials EARLY (like early stopping for hyperparameter search)
    # MedianPruner checks if the trial's intermediate score is below the median
    # of all completed trials at the same step — if so, it kills it early
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=10,   # don't prune until 10 trials are complete
        n_warmup_steps=2,      # don't prune until at least 2 CV folds are done
    )
)

# Run the optimization
# n_trials: how many total trials to run
# More trials = more computation but better results
# For text classification, 100 trials is a solid starting point
print("Starting Optuna optimization...")
study.optimize(
    objective,
    n_trials=100,
    timeout=3600,       # also stop after 1 hour if n_trials not reached
    show_progress_bar=True
)

print(f"\nBest trial: #{study.best_trial.number}")
print(f"Best F1 macro: {study.best_value:.4f}")
print("\nBest parameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Extract best params and rebuild the exact pipeline
best_params = study.best_params

# Reconstruct TF-IDF with best params
best_tfidf = TfidfVectorizer(
    max_features=best_params['tfidf_max_features'],
    ngram_range=best_params['tfidf_ngram_range'],
    min_df=best_params['tfidf_min_df'],
    sublinear_tf=best_params['tfidf_sublinear_tf'],
    max_df=0.95
)

# Reconstruct classifier with best params
if best_params['classifier'] == 'logreg':
    best_clf = LogisticRegression(
        C=best_params['logreg_C'],
        solver=best_params['logreg_solver'],
        class_weight='balanced',
        max_iter=1000,
        random_state=42
    )
else:
    best_clf = LinearSVC(
        C=best_params['svc_C'],
        class_weight='balanced',
        max_iter=2000,
        random_state=42
    )

# Final pipeline
best_pipeline = Pipeline([
    ('tfidf', best_tfidf),
    ('clf', best_clf)
])

# Fit on FULL training data (not just one fold)
best_pipeline.fit(X_train, y_train)

# Evaluate on held-out test set
y_pred = best_pipeline.predict(X_test)

print("=== OPTUNA-TUNED MODEL PERFORMANCE ===")
print(classification_report(y_test, y_pred, target_names=le.classes_))



In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
fig, ax = plt.subplots(figsize=(14, 12))
disp.plot(ax=ax, cmap='Blues', xticks_rotation=45)
plt.title("Confusion Matrix — Optuna Tuned Model")
plt.tight_layout()
plt.show()

In [ ]:
import joblib

joblib.dump(best_pipeline, 'ticket_classifier.joblib')
joblib.dump(le,'le.joblib')